# 🎯 DeepFace Attendance System — Google Colab
> Hệ thống chấm công nhận diện khuôn mặt với DeepFace + ArcFace

**Cấu trúc thư mục Google Drive cần chuẩn bị:**
```
MyDrive/
└── face_attendance/
    ├── dataset/          ← ảnh gốc (mỗi người 1 folder)
    │   ├── nguyen_van_a/
    │   │   ├── 001.jpg
    │   │   └── 002.jpg
    │   └── tran_thi_b/
    │       └── 001.jpg
    ├── augmented/        ← tự tạo khi chạy
    ├── embeddings/       ← tự tạo khi chạy
    └── attendance_log/   ← tự tạo khi chạy
```

## 📦 Bước 1: Cài thư viện

In [ ]:
!pip install deepface tf-keras opencv-python-headless albumentations openpyxl -q
!pip install tensorflow==2.15.0 -q
print('✅ Cài đặt hoàn tất!')

## 📁 Bước 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/face_attendance'
DATASET_DIR   = f'{BASE_DIR}/dataset'
AUGMENTED_DIR = f'{BASE_DIR}/augmented'
EMBED_DIR     = f'{BASE_DIR}/embeddings'
LOG_DIR       = f'{BASE_DIR}/attendance_log'

for d in [DATASET_DIR, AUGMENTED_DIR, EMBED_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Đã mount Google Drive và tạo thư mục!')

## 🔍 Bước 3: Kiểm tra dataset

In [ ]:
import glob
from collections import defaultdict

def check_dataset(dataset_dir):
    persons = [d for d in os.listdir(dataset_dir)
               if os.path.isdir(os.path.join(dataset_dir, d))]
    print(f'👥 Tổng số người: {len(persons)}')
    print('-' * 40)
    total = 0
    for p in sorted(persons):
        imgs = glob.glob(f'{dataset_dir}/{p}/*.jpg') + \
               glob.glob(f'{dataset_dir}/{p}/*.png') + \
               glob.glob(f'{dataset_dir}/{p}/*.jpeg')
        print(f'  {p}: {len(imgs)} ảnh', '⚠️ Cần thêm ảnh!' if len(imgs) < 5 else '✅')
        total += len(imgs)
    print('-' * 40)
    print(f'📸 Tổng số ảnh: {total}')
    return persons

persons = check_dataset(DATASET_DIR)

## 🔄 Bước 4: Data Augmentation
> Tự động tăng số lượng ảnh từ vài ảnh lên ~60-80 ảnh/người

In [ ]:
import cv2
import numpy as np
import albumentations as A
from tqdm import tqdm

# Pipeline augmentation cho ảnh khuôn mặt
augment_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.GaussNoise(var_limit=(10, 50), p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, p=0.5),
    A.Rotate(limit=15, p=0.7),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.CLAHE(p=0.3),                         # Tăng tương phản cục bộ
    A.RandomShadow(p=0.2),                  # Giả lập bóng đổ
    A.RandomGamma(p=0.3),
    A.Perspective(scale=(0.02, 0.05), p=0.3),
])

TARGET_IMAGES_PER_PERSON = 60  # Số ảnh mục tiêu mỗi người

def augment_person(person_name, dataset_dir, augmented_dir, target=60):
    src_dir = f'{dataset_dir}/{person_name}'
    dst_dir = f'{augmented_dir}/{person_name}'
    os.makedirs(dst_dir, exist_ok=True)

    # Copy ảnh gốc trước
    orig_imgs = glob.glob(f'{src_dir}/*.jpg') + \
                glob.glob(f'{src_dir}/*.png') + \
                glob.glob(f'{src_dir}/*.jpeg')

    if not orig_imgs:
        print(f'  ⚠️ {person_name}: Không tìm thấy ảnh!')
        return 0

    count = 0
    for img_path in orig_imgs:
        img = cv2.imread(img_path)
        if img is None:
            continue
        dst_path = f'{dst_dir}/orig_{count:04d}.jpg'
        cv2.imwrite(dst_path, img)
        count += 1

    # Augment thêm đến target
    aug_count = 0
    while count < target:
        src_img_path = orig_imgs[aug_count % len(orig_imgs)]
        img = cv2.imread(src_img_path)
        if img is None:
            aug_count += 1
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = augment_pipeline(image=img_rgb)['image']
        augmented_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
        cv2.imwrite(f'{dst_dir}/aug_{count:04d}.jpg', augmented_bgr)
        count += 1
        aug_count += 1

    return count

print('🔄 Đang augment dữ liệu...')
for person in tqdm(persons):
    n = augment_person(person, DATASET_DIR, AUGMENTED_DIR, TARGET_IMAGES_PER_PERSON)
    print(f'  ✅ {person}: {n} ảnh')

print('\n🎉 Augmentation hoàn tất!')
check_dataset(AUGMENTED_DIR)

## 🧠 Bước 5: Xây dựng Embedding Database
> Dùng model ArcFace — độ chính xác cao nhất trong DeepFace

In [ ]:
import pickle
from deepface import DeepFace

MODEL_NAME   = 'ArcFace'    # Tốt nhất cho chấm công
DETECTOR     = 'retinaface' # Tốt nhất cho phát hiện khuôn mặt
DISTANCE_METRIC = 'cosine'

def build_embedding_database(augmented_dir, embed_dir, model_name, detector):
    db = {}  # {person_name: [embedding_vectors]}
    persons = [d for d in os.listdir(augmented_dir)
               if os.path.isdir(os.path.join(augmented_dir, d))]

    print(f'🧠 Đang build embedding với {model_name} + {detector}...')

    for person in tqdm(persons):
        person_dir = f'{augmented_dir}/{person}'
        imgs = glob.glob(f'{person_dir}/*.jpg') + \
               glob.glob(f'{person_dir}/*.png')
        embeddings = []

        for img_path in imgs:
            try:
                result = DeepFace.represent(
                    img_path=img_path,
                    model_name=model_name,
                    detector_backend=detector,
                    enforce_detection=True,
                    align=True  # Căn chỉnh khuôn mặt — rất quan trọng!
                )
                if result:
                    embeddings.append(np.array(result[0]['embedding']))
            except Exception:
                # Bỏ qua ảnh không phát hiện được khuôn mặt
                pass

        if embeddings:
            db[person] = embeddings
            print(f'  ✅ {person}: {len(embeddings)} embeddings')
        else:
            print(f'  ❌ {person}: Không thể tạo embedding!')

    # Lưu database
    db_path = f'{embed_dir}/embeddings_{model_name.lower()}.pkl'
    with open(db_path, 'wb') as f:
        pickle.dump(db, f)

    print(f'\n💾 Đã lưu embedding database: {db_path}')
    return db, db_path

embedding_db, db_path = build_embedding_database(
    AUGMENTED_DIR, EMBED_DIR, MODEL_NAME, DETECTOR
)

## 🎯 Bước 6: Tìm ngưỡng (Threshold) tối ưu

In [ ]:
from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt

def compute_mean_embeddings(db):
    """Tính embedding trung bình cho mỗi người — tăng độ ổn định"""
    return {person: np.mean(embeds, axis=0)
            for person, embeds in db.items()}

def find_person(query_embedding, mean_db, threshold=0.4):
    """Nhận diện người từ embedding"""
    best_match = None
    best_dist  = float('inf')

    for person, mean_emb in mean_db.items():
        dist = cosine(query_embedding, mean_emb)
        if dist < best_dist:
            best_dist  = dist
            best_match = person

    if best_dist <= threshold:
        confidence = round((1 - best_dist) * 100, 2)
        return best_match, confidence, best_dist
    return 'UNKNOWN', 0.0, best_dist

# Tính mean embeddings
mean_db = compute_mean_embeddings(embedding_db)

# Visualize khoảng cách giữa các người
persons_list = list(mean_db.keys())
n = len(persons_list)
dist_matrix = np.zeros((n, n))

for i, p1 in enumerate(persons_list):
    for j, p2 in enumerate(persons_list):
        dist_matrix[i][j] = cosine(mean_db[p1], mean_db[p2])

fig, ax = plt.subplots(figsize=(max(6, n), max(5, n-1)))
im = ax.imshow(dist_matrix, cmap='RdYlGn_r', vmin=0, vmax=0.8)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(persons_list, rotation=45, ha='right')
ax.set_yticklabels(persons_list)
plt.colorbar(im, ax=ax, label='Cosine Distance (nhỏ = giống nhau)')
ax.set_title('Ma trận khoảng cách giữa các người\n(Đường chéo = 0, off-diagonal > 0.3 là tốt)')
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{dist_matrix[i][j]:.2f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{EMBED_DIR}/distance_matrix.png', dpi=150)
plt.show()
print('\n📊 Nếu off-diagonal > 0.3: ngưỡng 0.35-0.45 là phù hợp')
print('📊 Nếu có cặp < 0.3: cần thêm ảnh đa dạng hơn cho 2 người đó')

## 📸 Bước 7: Test nhận diện với ảnh mới

In [ ]:
from google.colab import files
from IPython.display import display, Image as IPImage
import io
from PIL import Image

THRESHOLD = 0.40  # Chỉnh ở đây nếu cần

def recognize_from_image(img_path, mean_db, threshold, model_name, detector):
    try:
        result = DeepFace.represent(
            img_path=img_path,
            model_name=model_name,
            detector_backend=detector,
            enforce_detection=True,
            align=True
        )
        if not result:
            return None, 0, 999

        query_emb = np.array(result[0]['embedding'])
        return find_person(query_emb, mean_db, threshold)

    except Exception as e:
        print(f'❌ Lỗi: {e}')
        return None, 0, 999

# Upload ảnh test
print('📤 Upload ảnh để test nhận diện:')
uploaded = files.upload()

for filename, data in uploaded.items():
    img_path = f'/tmp/{filename}'
    with open(img_path, 'wb') as f:
        f.write(data)

    person, confidence, dist = recognize_from_image(
        img_path, mean_db, THRESHOLD, MODEL_NAME, DETECTOR
    )

    display(IPImage(img_path, width=200))
    if person and person != 'UNKNOWN':
        print(f'✅ Nhận diện: {person}')
        print(f'   Độ tin cậy: {confidence}%  |  Distance: {dist:.4f}')
    elif person == 'UNKNOWN':
        print(f'❓ Không nhận diện được (distance={dist:.4f} > threshold={THRESHOLD})')
    else:
        print('❌ Không phát hiện khuôn mặt trong ảnh')

## 📋 Bước 8: Module Chấm Công (Attendance Logger)

In [ ]:
import pandas as pd
from datetime import datetime

def log_attendance(person_name, confidence, log_dir):
    """Ghi nhận chấm công vào file Excel theo ngày"""
    today = datetime.now().strftime('%Y-%m-%d')
    now   = datetime.now().strftime('%H:%M:%S')
    log_file = f'{log_dir}/attendance_{today}.xlsx'

    new_row = {
        'Họ tên': person_name,
        'Thời gian': now,
        'Ngày': today,
        'Độ tin cậy (%)': confidence,
        'Trạng thái': 'Có mặt'
    }

    if os.path.exists(log_file):
        df = pd.read_excel(log_file)
        # Kiểm tra đã chấm công chưa (tránh duplicate)
        if person_name in df['Họ tên'].values:
            print(f'ℹ️  {person_name} đã được chấm công hôm nay lúc {df[df["Họ tên"]==person_name]["Thời gian"].values[0]}')
            return False
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    else:
        df = pd.DataFrame([new_row])

    df.to_excel(log_file, index=False)
    print(f'✅ Đã chấm công: {person_name} lúc {now} ({confidence}%)')
    return True


def run_attendance(image_source, mean_db, threshold, model_name, detector, log_dir):
    """Pipeline chấm công hoàn chỉnh"""
    person, confidence, dist = recognize_from_image(
        image_source, mean_db, threshold, model_name, detector
    )

    if person is None:
        print('❌ Không phát hiện khuôn mặt')
        return

    if person == 'UNKNOWN':
        print(f'❓ Không nhận diện được (dist={dist:.3f})')
        return

    log_attendance(person, confidence, log_dir)


# Test attendance với ảnh upload
print('📤 Upload ảnh để chấm công:')
uploaded = files.upload()

for filename, data in uploaded.items():
    img_path = f'/tmp/{filename}'
    with open(img_path, 'wb') as f:
        f.write(data)
    run_attendance(img_path, mean_db, THRESHOLD, MODEL_NAME, DETECTOR, LOG_DIR)

## 📊 Bước 9: Xem báo cáo chấm công

In [ ]:
def view_attendance_report(log_dir, date=None):
    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')

    log_file = f'{log_dir}/attendance_{date}.xlsx'

    if not os.path.exists(log_file):
        print(f'📭 Chưa có dữ liệu chấm công ngày {date}')
        return

    df = pd.read_excel(log_file)
    print(f'📋 Báo cáo chấm công ngày {date}:')
    print(f'   Tổng có mặt: {len(df)} người')
    display(df)
    return df

view_attendance_report(LOG_DIR)

## 🔬 Bước 10 (Nâng cao): So sánh các Model
> Chạy cell này để tìm model tốt nhất cho dataset của bạn

In [ ]:
# So sánh ArcFace vs Facenet512 vs VGG-Face
MODELS_TO_COMPARE = ['ArcFace', 'Facenet512', 'VGG-Face']

results_summary = {}

for model in MODELS_TO_COMPARE:
    print(f'\n🧪 Testing {model}...')
    try:
        # Build embedding cho 5 ảnh đầu mỗi người (test nhanh)
        test_db = {}
        for person in persons:
            imgs = glob.glob(f'{AUGMENTED_DIR}/{person}/*.jpg')[:5]
            embeds = []
            for img_path in imgs:
                try:
                    r = DeepFace.represent(img_path, model_name=model,
                                          detector_backend='retinaface',
                                          enforce_detection=True, align=True)
                    if r:
                        embeds.append(np.array(r[0]['embedding']))
                except:
                    pass
            if embeds:
                test_db[person] = np.mean(embeds, axis=0)

        # Tính intra/inter distances
        intra_dists = []
        for person in persons:
            imgs = glob.glob(f'{AUGMENTED_DIR}/{person}/*.jpg')[5:10]
            for img_path in imgs:
                try:
                    r = DeepFace.represent(img_path, model_name=model,
                                          detector_backend='retinaface',
                                          enforce_detection=True, align=True)
                    if r and person in test_db:
                        d = cosine(np.array(r[0]['embedding']), test_db[person])
                        intra_dists.append(d)
                except:
                    pass

        avg_intra = np.mean(intra_dists) if intra_dists else 999
        results_summary[model] = avg_intra
        print(f'  Avg intra-class distance: {avg_intra:.4f} (nhỏ hơn = tốt hơn)')

    except Exception as e:
        print(f'  ❌ Lỗi: {e}')

print('\n🏆 Kết quả so sánh:')
for model, score in sorted(results_summary.items(), key=lambda x: x[1]):
    print(f'  {model}: {score:.4f} {"← Tốt nhất!" if model == min(results_summary, key=results_summary.get) else ""}')